# 财报智析 Agent 课程作业 Notebook

作者：徐北辰（202408240218，24级智能会计班）；周亦轩（202408240302，24级智能会计班）

本 Notebook 用于展示项目运行逻辑、核心代码调用和输出结果，并附小组分工、参考资料与独立完成声明。


## 1. 研究问题

本项目研究如何构建一个会计垂直 Agent，使其能够读取上市公司年报或标准财务数据，自动完成数据校验、指标计算、风险识别和报告生成。

问题来源于真实资本市场中的财报解读需求：深交所互动易、上证 e 互动、全景路演等平台上，投资者经常围绕经营现金流、应收账款、坏账准备、收入确认、资产负债率和财务指标变动原因向上市公司追问。相关文献也表明，年报可读性、文本复杂性和信息处理成本会影响投资者理解和风险识别。因此，本项目重点解决年报信息处理成本高、指标口径整理难、风险识别证据链弱和大模型直接生成财报结论容易幻觉的问题。

已有 CSMAR 智能财经报告分析平台、CSMAR 上市公司风险智能感知系统、弈 Chat、同花顺问财、Wind Alice、Choice 妙想 AI、Datayes AI 开放平台等相邻实践，说明 AI 财经问答、报告生成、风险诊断和 Agent 数据接口已经进入业务场景。本项目的差异是面向课堂与课程作业，强调本地可运行、确定性指标计算和 Agent trace 可追踪。


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


PosixPath('/Users/vison/Desktop/学习/大二下/生成式ai/期末作业/accounting-agent')

## 2. 读取样例数据


In [2]:
sample_path = PROJECT_ROOT / 'data' / 'sample_financial_data.xlsx'
raw_df = pd.read_excel(sample_path)
raw_df.head()


,项目,2022,2023,2024
0,营业收入,120000,145000,162000
1,营业成本,72000,92000,108000
2,净利润,14500,17800,16600
3,经营活动现金流量净额,16000,12500,9800
4,销售商品、提供劳务收到的现金,130000,150000,158000


## 3. 真实年报读取验证：科大讯飞 2025 年年度报告


In [3]:
real_case_dir = Path('evidence') / '科大讯飞2025年报案例'
if not real_case_dir.exists():
    real_case_dir = PROJECT_ROOT / 'outputs' / 'real_case' / 'keda_xunfei_2025'

case_summary = json.loads((real_case_dir / 'real_case_summary.json').read_text(encoding='utf-8'))
case_df = pd.read_csv(real_case_dir / 'extracted_financial_data.csv')
case_summary


{'source_pdf': 'outputs/uploads/科大讯飞：2025年年度报告.pdf',
 'extracted_csv': 'outputs/real_case/keda_xunfei_2025/extracted_financial_data.csv',
 'extracted_xlsx': 'outputs/real_case/keda_xunfei_2025/extracted_financial_data.xlsx',
 'audit_json': 'outputs/real_case/keda_xunfei_2025/extracted_financial_data_audit.json',
 'pages_scanned': 301,
 'pages_total': 301,
 'tables_found': 1073,
 'table_rows': 17921,
 'text_lines': 3804,
 'items_extracted': 15,
 'years': [2024, 2025],
 'coverage_ratio': 1.0,
 'missing_core_items': [],
 'validation_summary': '数据校验完成：通过 7 项，关注 0 项，异常 0 项。',
 'ratio_rows': 36,
 'risk_count': 1,
 'trace_steps': 13,
 'core_metrics': {'营业收入增长率': '16.12%',
  '净利润增长率': '67.54%',
  '资产负债率': '55.72%',
  '净资产收益率（ROE）': '4.40%',
  '净利润现金含量': '377.66%',
  '风险提示数量': '1'},
 'warnings': ['已剔除核心科目覆盖不足的年度：2023。这些年份多来自摘要或附注，未进入标准 CSV。']}

In [4]:
case_df


,项目,2024,2025
0,营业收入,2.334309e+10,2.710539e+10
1,净利润,5.070336e+08,8.494686e+08
2,非经常性损益,2.642769e+04,1.881373e+04
3,经营活动现金流量净额,2.495173e+09,3.208095e+09
4,货币资金,3.387380e+09,2.451176e+09
5,应收账款,1.466645e+10,1.629594e+10
6,存货,2.846519e+09,3.031004e+09
7,总资产,4.147890e+10,4.485711e+10
8,流动资产合计,2.406214e+10,2.542952e+10
9,流动负债合计,1.535905e+10,1.859505e+10


## 4. 运行 Agent 分析主流程


In [5]:
from agents.main_agent import run_financial_analysis
from config.settings import ModelConfig

result = run_financial_analysis(
    excel_path=sample_path,
    enable_pdf=False,
    generate_word=True,
    model_config_override=ModelConfig(enable_llm=False),
)
result.keys()


dict_keys(['options', 'model_config', 'raw_data', 'cleaned_data', 'missing_core_items', 'warnings', 'validation', 'ratio_results', 'ratio_pivot', 'trend_summary', 'dupont', 'cashflow', 'risks', 'charts', 'report', 'pdf_info', 'trace', 'trace_path', 'core_metrics'])

## 5. 查看数据校验结果


In [6]:
result['validation']['summary']
result['validation']['detail_table']


,年份,校验项目,校验结果,校验证据,处理建议
0,全部,核心科目完整性,通过,已识别收入、成本、利润、现金流、资产、负债和权益等核心科目。,可继续进入指标计算和风险识别。
1,2022,资产负债表勾稽关系,通过,"总资产 310,000.00；总负债+所有者权益 310,000.00；差异 0.00。差异...",若差异较大，请检查单位、合并/母公司口径和是否遗漏少数股东权益等项目。
2,2022,利润表基础合理性,通过,营业收入、营业成本和净利润未触发基础异常校验。,可继续结合盈利能力指标和趋势变化判断利润质量。
3,2022,利润现金流匹配性,通过,净利润与经营活动现金流量净额未触发基础背离校验。,仍需结合净利润现金含量和营运资金占用进一步判断。
4,2023,资产负债表勾稽关系,通过,"总资产 365,000.00；总负债+所有者权益 365,000.00；差异 0.00。差异...",若差异较大，请检查单位、合并/母公司口径和是否遗漏少数股东权益等项目。
5,2023,利润表基础合理性,通过,营业收入、营业成本和净利润未触发基础异常校验。,可继续结合盈利能力指标和趋势变化判断利润质量。
6,2023,利润现金流匹配性,通过,净利润与经营活动现金流量净额未触发基础背离校验。,仍需结合净利润现金含量和营运资金占用进一步判断。
7,2024,资产负债表勾稽关系,通过,"总资产 430,000.00；总负债+所有者权益 430,000.00；差异 0.00。差异...",若差异较大，请检查单位、合并/母公司口径和是否遗漏少数股东权益等项目。
8,2024,利润表基础合理性,通过,营业收入、营业成本和净利润未触发基础异常校验。,可继续结合盈利能力指标和趋势变化判断利润质量。
9,2024,利润现金流匹配性,通过,净利润与经营活动现金流量净额未触发基础背离校验。,仍需结合净利润现金含量和营运资金占用进一步判断。


## 6. 查看财务指标表


In [7]:
result['ratio_pivot'].head(20)


,指标类别,指标名称,计算公式,指标含义,2022,2023,2024
0,偿债能力,权益乘数,总资产 / 所有者权益,衡量权益资本撬动资产规模的杠杆倍数。,1.70,1.87,2.10
1,偿债能力,流动比率,流动资产 / 流动负债,衡量企业短期偿债能力。,1.97,1.81,1.66
2,偿债能力,资产负债率,总负债 / 总资产,衡量企业财务杠杆水平。,41.29%,46.58%,52.33%
3,偿债能力,速动比率,(流动资产 - 存货) / 流动负债,剔除存货后衡量即时短期偿债能力。,1.47,1.25,0.98
4,成长能力,净利润增长率,(本年净利润 - 上年净利润) / 上年净利润,衡量利润成长速度。,,22.76%,-6.74%
5,成长能力,总资产增长率,(本年总资产 - 上年总资产) / 上年总资产,衡量资产规模扩张速度。,,17.74%,17.81%
6,成长能力,所有者权益增长率,(本年所有者权益 - 上年所有者权益) / 上年所有者权益,衡量权益资本积累速度。,,7.14%,5.13%
7,成长能力,营业收入增长率,(本年营业收入 - 上年营业收入) / 上年营业收入,衡量收入规模成长速度。,,20.83%,11.72%
8,现金流量质量,净利润现金含量,经营活动现金流量净额 / 净利润,衡量净利润由经营活动现金流量净额支撑的程度。,110.34%,70.22%,59.04%
9,现金流量质量,经营现金流量收入比率,经营活动现金流量净额 / 营业收入,衡量营业收入转化为经营活动现金流量净额的能力。,13.33%,8.62%,6.05%


## 7. 查看风险识别结果


In [8]:
pd.DataFrame(result['risks'])


,risk_id,risk_name,risk_level,evidence,explanation,suggestion
0,R101,应收账款增长异常,中,2024 年应收账款增长率 51.22%，高于营业收入增长率 11.72%。,提示赊销规模或回款压力可能上升，收入质量需结合账龄结构进一步核查。,关注客户信用政策、应收账款账龄和坏账准备计提充分性。
1,R102,存货增长较快,中,2024 年存货增长率 46.15%，高于营业成本增长率 17.39%。,提示存货周转和跌价压力可能增加。,进一步核查存货库龄、产品结构和跌价准备计提情况。
2,R104,毛利率连续下降,中,毛利率近三年连续下降。,提示产品竞争力或成本控制可能承压。,关注售价、原材料成本、产品结构和费用转嫁能力。
3,R105,资产负债率持续上升,中,资产负债率近三年持续上升。,提示偿债压力和财务弹性需关注。,进一步区分金融性负债与经营性负债，并关注债务期限结构。
4,R106,短期偿债能力下降,中,2024 年流动比率和速动比率较上年同时下降。,提示短期偿债安全边际可能收窄。,关注流动负债到期压力、受限资金和存货变现能力。
5,R107,经营活动现金流量净额长期低于净利润,高,净利润现金含量至少两个年度低于 100%。,提示盈余质量和利润现金含量需关注。,结合商业信用、应收账款回款和存货占用进一步核查。
6,R109,资产减值损失大幅增加,中,2024 年资产减值相关项目增长 72.22%。,提示资产质量可能承压，需关注减值测试和相关资产可收回金额。,进一步核查应收、存货、商誉和长期资产减值明细。
7,R110,非经常性损益影响较大,中,2024 年非经常性损益占净利润比例约 25.30%。,提示利润可持续性需关注。,重点比较扣非净利润和经常性业务盈利能力。


## 8. 查看 Agent 调用记录


In [9]:
pd.DataFrame(result['trace'])


,order,tool_name,action,input_summary,output_summary,status,elapsed_seconds,error
0,1,llm_model,加载大模型 API 配置,openai_compatible/deepseek-v4-flash,未配置 DEEPSEEK_API_KEY，将使用 Python fallback pipel...,完成,0.000,
1,2,main_agent,smolagents 主控 Agent 规划工具调度,标准财务数据 + 可选 PDF,未配置 API Key，使用固定工具链完成分析。,完成,0.000,
2,3,file_loader,读取标准财务数据 CSV/Excel,/Users/vison/Desktop/学习/大二下/生成式ai/期末作业/account...,读取 15 行、4 列。,完成,0.002,
3,4,pdf_parser,解析年度报告 PDF,,未启用或未上传 PDF。,跳过,0.000,
4,5,data_cleaner,标准化财务报表项目,识别宽表/长表和中文科目,识别 15 个标准科目、3 个年度。,完成,0.008,
5,6,data_validator,校验数据完整性与勾稽关系,核心科目 + 资产负债表等式 + 利润现金流匹配,数据校验完成：通过 10 项，关注 0 项，异常 0 项。,完成,0.006,
6,7,ratio_calculator,计算财务指标,偿债、营运、盈利、成长、现金流量质量,生成 54 条指标记录。,完成,0.005,
7,8,trend_analyzer,分析多年度趋势,财务项目和核心指标,生成 14 条趋势判断。,完成,0.005,
8,9,dupont_analyzer,执行杜邦分析,净资产收益率（ROE）= 净利率 × 总资产周转率 × 权益乘数,2023 年至 2024 年净资产收益率（ROE）下降，变化主要受权益乘数影响。,完成,0.002,
9,10,cashflow_analyzer,分析现金流量质量,经营活动现金流量净额与净利润匹配,现金流量质量评价：风险较高,完成,0.000,


## 9. 输出文件

- Word 报告：`outputs/reports/financial_analysis_report.docx`
- Markdown 报告：`outputs/reports/financial_analysis_report.md`
- 指标表：`outputs/tables/ratio_results.xlsx`
- 数据校验表：`outputs/tables/data_validation.xlsx`
- Agent trace：`outputs/agent_trace.json`


## 10. 附录：小组贡献、参考文献与声明

### 小组贡献说明

| 成员 | 学号 | 班级 | 主要贡献 | 贡献比例 |
|---|---|---|---|---|
| 徐北辰 | 202408240218 | 24级智能会计班 | 负责智能体总体架构设计、smolagents 主控 Agent 接入、后端工具链组织、Streamlit 前端工作台构建、API 配置区、动态 Agent trace、数据读取与分析流程串联、代码调试、测试验证和最终工程打包。 | 60% |
| 周亦轩 | 202408240302 | 24级智能会计班 | 负责财务报表分析教材框架、指标体系、风险规则和相关文献资料的收集整理，参与报告结构设计、页面表达优化和前端视觉美化，协助校对指标名称、报告表述和课堂展示材料。 | 40% |

### 学习心得

- 徐北辰：徐北辰在项目中主要负责把会计分析任务拆解为可执行、可追踪的 Agent 工具链。在查阅投资者互动问答和现有智能财经平台后，进一步认识到财报分析的难点不是简单生成一段结论，而是要把年报数字、指标口径、风险触发依据和报告表达连接成可复核的证据链。因此在搭建过程中，将财务数据清洗、指标计算、勾稽校验和风险识别交给确定性 Python 模块完成，让大模型承担计划生成和语言组织。通过前端工作台和动态 trace 的实现，也体会到一个可演示的 Agent 系统不仅要能算，还要让用户看清楚它正在调用什么工具、依据什么数据、输出什么结果。
- 周亦轩：周亦轩在项目中主要围绕会计专业资料和展示体验展开工作。通过整理教材框架、指标定义和风险规则，更加清楚地理解了传统财务报表分析中偿债能力、营运能力、盈利能力、成长能力、现金流量质量和杜邦分析之间的逻辑关系。在补充年报可读性、网络平台互动、信息处理成本和大数据审计等文献后，也进一步理解了财务报告分析工具的专业价值：它需要帮助用户降低阅读成本、统一指标口径，并把风险提示限定在审慎、可追溯的范围内。在参与前端美化和报告结构设计时，也认识到会计智能体的展示界面需要兼顾专业性和可读性，既要保持财务系统的稳重风格，也要让用户能够快速找到数据来源、指标结论和风险依据。

### 参考文献与资料

- 张新民、钱爱民：《财务报表分析（第6版·立体化数字教材版）案例分析与学习指导》，中国人民大学出版社，ISBN 978-7-300-31977-3。
- 谭松涛, 阚铄, 崔小勇. 互联网沟通能够改善市场信息效率吗?——基于深交所“互动易”网络平台的研究[J]. 金融研究, 2016(3):174-188.
- 徐寿福, 郑迎飞, 罗雨杰. 网络平台互动与股票异质性风险[J]. 财经研究, 2022, 48(10):153-168.
- 周波, 李洁柔, 王少飞. 整合信息披露、信息处理成本与投资意愿——基于个体投资者判断的实验研究[J]. 财经研究, 2025, 51(4):139-154.
- 徐巍, 姚振晔, 陈冬华. 中文年报可读性:衡量与检验[J]. 会计研究, 2021(03):28-44.
- 王克敏, 王华杰, 李栋栋, 戴杏云. 年报文本信息复杂性与管理者自利——来自中国上市公司的证据[J]. 管理世界, 2018, 34(12):120-132.
- 郭松林, 宁祺器, 窦斌. 上市公司年报文本增量信息与违规风险预测——基于语调和可读性的视角[J]. 统计研究, 2022, 39(12):69-84.
- 徐荣华, 朱婧, 戴欣瑜. 大数据审计:理论框架、研究进展与未来展望[J]. 外国经济与管理, 2024, 46(11):122-137.
- 深交所互动易: https://irm.cninfo.com.cn/
- 上证 e 互动: https://sns.sseinfo.com/
- 全景路演投资者关系互动平台: https://ir.p5w.net/
- CSMAR 智能财经报告分析平台: https://www.csmar.com/channels/63.html
- CSMAR 上市公司风险智能感知系统: https://www.csmar.com/channels/上市公司风险智能感知系统.html
- CSMAR 弈 Chat 智能财经对话系统: https://www.csmar.com/channels/102.html
- 同花顺问财: https://www.iwencai.com/
- Wind 金融终端与 Wind Alice: https://www.wind.com.cn/
- Choice 智能金融终端: https://choice.eastmoney.com/
- Datayes AI 开放平台: https://d.datayes.com/
- Hugging Face smolagents 文档: https://huggingface.co/docs/smolagents
- DeepSeek API 文档: https://api-docs.deepseek.com/
- Streamlit 文档: https://docs.streamlit.io/
- pandas 文档: https://pandas.pydata.org/docs/
- pdfplumber、PyMuPDF、python-docx 等开源工具文档。

### 独立完成声明

本课程作业由徐北辰、周亦轩在课程要求范围内独立完成。签名：徐北辰、周亦轩。日期：2026年6月12日。
